# Tutorial 3 — End-to-End with a Quantum Framework

<a href="https://colab.research.google.com/github/quprep/quprep/blob/main/examples/tutorials/03_end_to_end_with_a_framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/quprep/quprep/v0.10.0?labpath=examples%2Ftutorials%2F03_end_to_end_with_a_framework.ipynb)
<a href="https://account.qbraid.com?gitHubUrl=https://github.com/quprep/quprep/blob/main/examples/tutorials/03_end_to_end_with_a_framework.ipynb"><img src="https://qbraid-static.s3.amazonaws.com/logos/Launch_on_qBraid_white.png" height="23"></a>


You have a dataset and you're not sure which encoder to use, how many qubits you need, or how to structure the preprocessing pipeline. QuPrep can answer all three questions automatically — and then produce circuits that go straight into Qiskit, PennyLane, or any other framework.

This tutorial shows the **intelligent path**: let QuPrep audit your data, suggest a pipeline, build it, verify the output, and export ready-to-use QASM circuits.

**No optional dependencies required.**

---
**What you will learn:**
- Auditing a dataset with `preprocessing_report()`
- Getting an automatic pipeline suggestion with `suggest_pipeline()`
- Building and running the pipeline
- Post-encoding verification with `verify_encoding()`
- Exporting to OpenQASM 3.0

In [ ]:
!pip install -q quprep scikit-learn

In [ ]:
import warnings

import numpy as np
from sklearn.datasets import load_breast_cancer

import quprep as qd
from quprep import QuPrepWarning

print(f"quprep {qd.__version__}")

## Step 1 — Load a dataset

Breast cancer classification: 569 samples, 30 features, binary target. We use 60 samples and 6 features to keep the output readable — the same workflow scales to the full dataset.

In [ ]:
bc = load_breast_cancer()
X_raw = bc.data[:60, :6]
y     = bc.target[:60]

dataset = qd.NumpyIngester().load(X_raw, y=y)

print(f"Samples  : {dataset.data.shape[0]}")
print(f"Features : {dataset.data.shape[1]}")
print(f"Classes  : {np.bincount(y.astype(int))}")
print(f"Range    : [{dataset.data.min():.2f}, {dataset.data.max():.2f}]")

## Step 2 — Audit the dataset

`preprocessing_report()` runs a structured audit before you touch a single hyperparameter. It checks for NaN, outliers, qubit budget overruns, class imbalance (>3:1), and encoder compatibility — and returns a list of actionable recommendations.

Run this first, every time. It takes milliseconds and saves debugging hours.

In [ ]:
report = qd.preprocessing_report(
    dataset,
    encoder=qd.AngleEncoder(),
    qubit_budget=6,
)
print(f"Issues : {report.n_issues}")
if report.recommendations:
    for rec in report.recommendations:
        print(f"  → {rec}")
else:
    print("  None — data looks healthy for this encoder")

## Step 3 — Get an automatic pipeline suggestion

`suggest_pipeline()` analyses the dataset and returns a `PipelineSuggestion` — a recommended combination of imputer, normalizer, and encoder — with a plain-English reason for each choice.

This is a starting point, not a black box. Override any component you like.

In [ ]:
suggestion = qd.suggest_pipeline(dataset, task="classification", qubits=6)

print(f"Encoder         : {suggestion.encoder}")
print(f"Normalizer      : {suggestion.normalizer}")
print(f"Imputer         : {suggestion.imputer or 'none needed'}")
print(f"Outlier handler : {suggestion.outlier_handler or 'none needed'}")
print(f"Reasoning       : {suggestion.reason}")

## Step 4 — Build and run the pipeline

`suggestion.build()` constructs a ready-to-use `Pipeline` from the suggestion. You can also build it manually if you want to customise individual components.

In [ ]:
pipeline = suggestion.build()

with warnings.catch_warnings():
    warnings.simplefilter("ignore", QuPrepWarning)
    result = pipeline.fit_transform(dataset)

encoder_instance = pipeline.encoder  # keep a reference for verify_encoding

print(f"Circuits : {len(result.encoded)}")
print(f"Qubits   : {result.encoded[0].metadata.get('n_qubits')}")
print(f"Depth    : {result.encoded[0].metadata.get('depth')}")

## Step 5 — Verify the encoding

`verify_encoding()` checks post-encoding invariants: for angle encoders, that all values are within the valid rotation range; for amplitude encoders, that state vectors have unit norm. Run it as a final sanity check before handing circuits to hardware.

In [ ]:
verify_report = qd.verify_encoding(result.encoded, encoder_instance)

print(f"Passed : {verify_report.passed}")
for check in verify_report.checks:
    status = "✓" if check["passed"] else "✗"
    print(f"  {status} {check['name']} : {check['detail']}")

## Step 6 — Export to OpenQASM 3.0

OpenQASM 3.0 is accepted by Qiskit, PennyLane, Cirq, and most simulators. For native framework objects (Qiskit `QuantumCircuit`, PennyLane tapes), see **how-to/export_to_frameworks**.

In [ ]:
exporter = qd.QASMExporter()
qasm = exporter.export(result.encoded[0])
print(qasm)

## Step 7 — ASCII circuit diagram

In [ ]:
print(qd.draw_ascii(result.encoded[0]))

---
## What just happened?

| Step | What QuPrep did |
|------|----------------|
| `preprocessing_report()` | Audited dataset for NaN, outliers, budget, imbalance, and encoder compatibility |
| `suggest_pipeline()` | Chose imputer, normalizer, and encoder using dataset heuristics |
| `suggestion.build()` | Assembled a `Pipeline` from the suggestion |
| `verify_encoding()` | Confirmed all circuits satisfy the encoder's invariants |
| `QASMExporter` | Produced OpenQASM 3.0 strings ready for any framework |

## Where to go next

| Guide | What you'll learn |
|-------|------------------|
| [how-to/choose_an_encoder](../how-to/choose_an_encoder.ipynb) | Compare all 15 encoders for your task |
| [how-to/export_to_frameworks](../how-to/export_to_frameworks.ipynb) | Export to PennyLane, Cirq, TKET, Braket |
| [how-to/validate_before_encoding](../how-to/validate_before_encoding.ipynb) | Deep-dive into `check_compatibility` and `DataSchema` |
| [how-to/assess_encoding_quality](../how-to/assess_encoding_quality.ipynb) | Expressibility, entanglement, barren plateau risk |